In [21]:
import sys
sys.path.append("/host/d/Github/")
import os
import numpy as np
import pandas as pd
import nibabel as nb
import re
import json
import shutil
import Osteosarcoma.functions_collection as ff 
import Osteosarcoma.Data_processing as Data_processing

In [22]:
file_list = ff.find_all_target_files(['Tumor*'],os.path.join('/host/d/projects/Habitats/segmentation/models/Dataset602_Tumor/results/predicts_raw/fold_0'))

In [23]:
import numpy as np
from scipy import ndimage

def remove_scatter(mask3d: np.ndarray,
                   target_label: int = 1,
                   connectivity: int = 26,
                   return_is_single: bool = False):
    """
    Keep only the largest 3D connected component of voxels == target_label.

    Args:
        mask3d: 3D numpy array, e.g. (X,Y,Z) with integer labels.
        target_label: which label to process (default 1).
        connectivity: 6, 18, or 26 for 3D connectivity.
        return_is_single: if True, also return whether it was a single component.

    Returns:
        cleaned_mask: same shape as mask3d, only largest component kept for target_label.
        (optional) is_single: bool, whether original target_label voxels form exactly 1 component.
        (optional) n_components: int, number of connected components (excluding background).
    """
    assert mask3d.ndim == 3, f"mask3d must be 3D, got {mask3d.ndim}D"

    # binary mask for the target label
    bin_mask = (mask3d == target_label)

    # if empty, return as-is
    if not np.any(bin_mask):
        if return_is_single:
            return mask3d.copy(), True, 0
        return mask3d.copy()

    # choose connectivity structure
    if connectivity == 6:
        structure = ndimage.generate_binary_structure(3, 1)
    elif connectivity == 18:
        structure = ndimage.generate_binary_structure(3, 2)
        # generate_binary_structure(3,2) actually gives 18-neighborhood in 3D
    elif connectivity == 26:
        structure = np.ones((3, 3, 3), dtype=bool)
    else:
        raise ValueError("connectivity must be one of {6, 18, 26}")

    labeled, ncomp = ndimage.label(bin_mask, structure=structure)

    # 判断是否单一连通域
    is_single = (ncomp == 1)

    if ncomp <= 1:
        cleaned = mask3d.copy()
        if return_is_single:
            return cleaned, is_single, ncomp
        return cleaned

    # compute component sizes (exclude background label 0)
    sizes = ndimage.sum(bin_mask, labeled, index=np.arange(1, ncomp + 1))
    largest_cc = int(np.argmax(sizes) + 1)

    largest_mask = (labeled == largest_cc)

    # build output: keep only largest CC for target_label, keep other labels unchanged
    cleaned = mask3d.copy()
    # remove all target_label first
    cleaned[bin_mask] = 0
    # put back the largest CC as target_label
    cleaned[largest_mask] = target_label

    if return_is_single:
        return cleaned, is_single, ncomp
    return cleaned


In [24]:
for i in range(0, len(file_list)):
    file_path = file_list[i]
    
    img = nb.load(file_path)
    img_data = img.get_fdata()

    final_result, is_single ,_ = remove_scatter(img_data, target_label=1, connectivity=26, return_is_single=True)

    if is_single != True:
        nb.save(nb.Nifti1Image(final_result, img.affine), file_path)
    

### save with images

In [26]:
patient_list_set2 = pd.read_excel('/host/d/Data/Habitats/Jishuitan/Patient_lists/image_info_set2.xlsx')
patient_list_set2 = patient_list_set2[patient_list_set2['Have_seg']!='Yes']
patient_set_list_set2 = patient_list_set2['Patient_set']
patient_index_list_set2 = patient_list_set2['Patient_index']
print('len:  ', len(patient_list_set2))
# phase_list_set2 = ['Tr' if patient_list_set2.iloc[i]['Have_seg'] == 'Yes' else 'Ts' for i in range(len(patient_list_set2))]

len:   153


In [27]:
import shutil
data_folder = os.path.join('/host/d/projects/Habitats/segmentation/models/Dataset602_Tumor/results/predicts_raw/fold_0')
save_folder = os.path.join('/host/d/projects/Habitats/segmentation/models/Dataset602_Tumor/results/predicts')
for i in range(len(patient_list_set2)):
    patient_set = patient_set_list_set2.iloc[i]
    patient_index = patient_index_list_set2.iloc[i]
    patient_name = 'Tumor_'+patient_set +'_' + str(patient_index).zfill(4)
    
    save_patient_folder = os.path.join(save_folder,patient_set, str(patient_index))
    ff.make_folder([os.path.dirname(save_patient_folder), save_patient_folder])

    # copy
    pred_file = os.path.join(data_folder, patient_name + '.nii.gz')
    shutil.copy(pred_file, os.path.join(save_patient_folder, 'pred_label.nii.gz'))
